# Notebook 01 — Ingestion pipeline
**Day 1 goal:** YouTube URL → transcript → chunks → Pinecone

By the end of this notebook you will have:
- A transcript pulled from a real YouTube video
- That transcript split into overlapping chunks
- Those chunks embedded and upserted to your Pinecone index

Run cells top to bottom. If a cell fails, read the error — it usually means a missing env var or a video with no captions.

## Step 1 — Environment check
Make sure your `.env` is configured before running anything else.

In [ ]:
# Ensure required packages are available in this kernel
import sys, subprocess
required = {
    'youtube-transcript-api': 'youtube_transcript_api',
    'langchain-text-splitters': 'langchain_text_splitters',
    'langchain-openai': 'langchain_openai',
    'pinecone': 'pinecone',
    'python-dotenv': 'dotenv',
    'openai': 'openai',
}
missing = []
for pkg, mod in required.items():
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    print('Installing:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing)
    print('Installed — restart kernel if imports still fail.')
else:
    print('All required packages already installed')

In [ ]:
import sys
sys.path.append('..')  # so we can import from src/

from src.utils.config import (
    OPENAI_API_KEY,
    PINECONE_API_KEY,
    PINECONE_INDEX_NAME,
    LANGCHAIN_API_KEY,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
)

print('✅ OpenAI key loaded:', OPENAI_API_KEY[:8] + '...')
print('✅ Pinecone key loaded:', PINECONE_API_KEY[:8] + '...')
print('✅ LangSmith key loaded:', LANGCHAIN_API_KEY[:8] + '...')
print(f'✅ Pinecone index: {PINECONE_INDEX_NAME}')
print(f'✅ Chunk size: {CHUNK_SIZE}, overlap: {CHUNK_OVERLAP}')

## Step 2 — Pinecone connection test
Upsert a single dummy vector and query it back. Confirms your index is live.

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

# Upsert one dummy vector (avoid all zeros — Pinecone rejects them)
dummy_vector = [1e-6] * 1536
index.upsert(vectors=[{'id': 'test-000', 'values': dummy_vector, 'metadata': {'text': 'test'}}])
print('✅ Upsert succeeded')

# Query it back
result = index.query(vector=dummy_vector, top_k=1, include_metadata=True)
print('✅ Query succeeded:', result['matches'][0]['id'])

# Delete the dummy
index.delete(ids=['test-000'])
print('✅ Pinecone connection confirmed — dummy vector cleaned up')

stats = index.describe_index_stats()
print(f'\nIndex stats: {stats["total_vector_count"]} vectors currently stored')

## Step 3 — Fetch a transcript
Paste any YouTube URL below. Use a skincare or influencer video for a realistic test.

In [ ]:
from src.ingestion.transcript import extract_video_id, fetch_transcript

# ← Change this to any YouTube URL
TEST_URL = 'https://www.youtube.com/watch?v=8Uf8HaUMmyk'

video_id = extract_video_id(TEST_URL)
print(f'Video ID: {video_id}')

transcript = fetch_transcript(video_id)
print(f'\nTranscript length: {len(transcript)} characters')
print(f'\nFirst 500 chars:\n{transcript[:500]}')

## Step 4 — Chunk the transcript
Split into overlapping chunks with metadata attached.

In [ ]:
from src.ingestion.transcript import chunk_transcript

metadata = {
    'video_id': video_id,
    'url': TEST_URL,
    'title': 'Test video',
}

chunks = chunk_transcript(transcript, metadata)

print(f'Total chunks: {len(chunks)}')
print(f'\nChunk 0 ({len(chunks[0]["text"])} chars):')
print(chunks[0]['text'])
print(f'\nMetadata: {chunks[0]["metadata"]}')

## Step 5 — Embed one chunk (cost check)
Before upserting everything, embed a single chunk to verify the embedding model is working and check the vector dimension.

In [ ]:
from src.ingestion.embedder import embed_texts

sample_embedding = embed_texts([chunks[0]['text']])[0]
print(f'✅ Embedding dimension: {len(sample_embedding)}')
print(f'First 5 values: {sample_embedding[:5]}')
assert len(sample_embedding) == 1536, 'Dimension mismatch! Check OPENAI_EMBEDDING_MODEL in .env'

## Step 6 — Full ingest (one video)
Run the complete pipeline: transcript → chunks → embed → upsert.

In [ ]:
from src.ingestion.embedder import ingest_video

summary = ingest_video(TEST_URL, title='Test video — notebook 01')
print('Ingestion summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

## Step 7 — Verify in Pinecone
Confirm the vectors landed in the index.

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
stats = index.describe_index_stats()

print(f'Total vectors in index: {stats["total_vector_count"]}')
print('\n✅ Day 1 complete. Move to notebook 02 to build the RAG chain.')

## Notes

**If `fetch_transcript` fails:**  
The video may not have auto-generated captions. Try a different video. Whisper transcription is a Phase 2 feature.

**Pinecone free tier limits:**  
1 index, ~100k vectors. Each video generates roughly 20–60 chunks depending on length. You have room for 1000+ videos before hitting the limit.

**Re-ingesting the same video:**  
Safe — chunk IDs are deterministic (MD5 of `video_id + chunk_index`), so Pinecone will overwrite existing vectors rather than creating duplicates.